# Model Evaluation & Optimization

Avalia os modelos segundo as métricas relevantes para deteção de malware. A comparação e a afinação usam apenas treino; a validação escolhe o limiar para a classe maliciosa; o teste fica reservado para a avaliação final.

In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd

SRC_PATH = str(Path("../src").resolve())
sys.path.insert(0, SRC_PATH)
sys.modules.pop("data_preparation", None)
sys.modules.pop("data_modelling", None)
sys.modules.pop("data_evaluation", None)

import data_preparation as dp
import data_modelling as dm
import data_evaluation as de

## 1. Preparar os dados

Reproduz a divisão estratificada definida em Data Preparation. Neste projeto, `0` corresponde a benigno e `1` a malicioso.

In [ ]:
DATASET_NAME = "malware"
DATA_PATH = "../dataset/raw/Malware_and_benign_recognition.csv"
OUTPUT_DIR = "../models"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
POSITIVE_LABEL = 1

X_train, X_val, X_test, y_train, y_val, y_test = dp.run_data_preparation(
    DATA_PATH, target="Malicious", random_state=42,
)
if set(y_train.unique()) != {0, 1}:
    raise ValueError("Este notebook espera a variável Malicious codificada como 0/1.")

## 2. Selecionar e afinar o modelo

A comparação usa F1 macro em validação cruzada estratificada. A pesquisa de hiperparâmetros do modelo selecionado também usa apenas treino; a validação não participa nesta escolha.

In [ ]:
results_df = dm.evaluate_models(X_train, y_train, scoring="f1_macro")
dm.save_results_csv(
    results_df, dataset_name=DATASET_NAME, output_dir=OUTPUT_DIR,
    timestamp=RUN_TIMESTAMP,
)
results_df

In [ ]:
best_name = dm.select_best_model(results_df)
search = de.tune_hyperparameters(
    best_name, X_train, y_train, scoring="f1_macro",
    n_splits=5, random_state=42,
)
best_pipe = search.best_estimator_
cv_tuning_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
cv_tuning_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]]

## 3. Escolher o limiar na validação

O limiar maximiza o F1 da classe maliciosa na validação. O modelo permanece ajustado apenas em treino para que a validação continue independente do ajuste do estimador. O resultado deve ser interpretado em conjunto com recall e falsos negativos.

In [ ]:
val_scores, score_type = de.get_positive_class_scores(
    best_pipe, X_val, positive_label=POSITIVE_LABEL,
)
threshold_result = de.optimize_threshold(
    y_val, val_scores, positive_label=POSITIVE_LABEL,
)
selected_threshold = threshold_result["selected"]["threshold"]
threshold_result["selected"]

## 4. Avaliação final no teste

O conjunto de teste é usado depois de fixados o modelo, hiperparâmetros e limiar. São reportados o limiar padrão e o limiar escolhido na validação; só o segundo é a configuração otimizada.

In [ ]:
test_results = de.evaluate_test_thresholds(
    best_pipe, X_test, y_test, selected_threshold, score_type,
    positive_label=POSITIVE_LABEL, dataset_name=DATASET_NAME,
    output_dir=OUTPUT_DIR, timestamp=RUN_TIMESTAMP,
)
pd.DataFrame([
    {"configuracao": "limiar_padrao", **{k: v for k, v in test_results["baseline"].items() if k not in ("classification_report", "confusion_matrix")}},
    {"configuracao": "limiar_otimizado", **{k: v for k, v in test_results["optimized"].items() if k not in ("classification_report", "confusion_matrix")}},
])

## 5. Guardar avaliação e modelo

Os artefactos ficam em `models/` com timestamp. O pipeline guardado permanece ajustado apenas no treino, compatível com o limiar escolhido na validação; o JSON inclui parâmetros, métricas e limiar para utilização posterior.

In [ ]:
evaluation_metrics = {
    "dataset": DATASET_NAME,
    "model": best_name,
    "positive_class": "Malicioso (1)",
    "n_splits": 5,
    "cv_best_f1_macro": float(search.best_score_),
    "best_params": search.best_params_,
    "threshold_selection": {
        **threshold_result["selected"],
        "score_type": score_type,
        "objective": "maximizar F1 da classe maliciosa na validacao",
    },
    "test": test_results,
}
artifact_paths = de.save_evaluation_artifacts(
    evaluation_metrics, threshold_result["curve"],
    dataset_name=DATASET_NAME, output_dir=OUTPUT_DIR, timestamp=RUN_TIMESTAMP,
)
model_path = dm.save_model(
    best_pipe, best_name, dataset_name=DATASET_NAME, output_dir=OUTPUT_DIR,
    metrics=evaluation_metrics, feature_names=list(X_train.columns),
    timestamp=RUN_TIMESTAMP,
)
print(f"Modelo guardado em: {model_path}")
artifact_paths